# 01 - Data Understanding

- Doc LCDataDictionary, lam ro cac truong dung duoc tai thoi diem xet duyet
- Load accepted + rejected raw
- Doi chieu schema accepted vs. rejected (RQ5)
- Ap dung filter_vintage() + drop_leakage_columns(), luu ra data/interim

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.paths import DATA_INTERIM, DATA_DICTIONARY
from src.data.load import load_accepted_vintage, load_rejected_vintage
from src.data.filter_vintage import LEAKAGE_COLUMNS

## Data dictionary

In [ ]:
data_dict = pd.read_excel(DATA_DICTIONARY, sheet_name=0)
data_dict.columns = [c.strip() for c in data_dict.columns]
data_dict.head(20)

## Load + loc vintage + loai bien leakage

Accepted goc la 2.26M dong x 151 cot (~3.3GB neu load full). Sau khi loc vintage 2015-2017, term 36 thang, `loan_status` in {Fully Paid, Charged Off} (PROPOSAL.md muc 4.2) chi con ~640K dong. `load_accepted_vintage()` loc va loai bien hau-giai-ngan (leakage, muc 4.3) ngay tren tung chunk khi doc, tranh phai giu ca 151 cot x 2.26M dong trong memory cung luc voi rejected.

In [ ]:
print('leakage columns bi loai:', LEAKAGE_COLUMNS)
accepted_clean = load_accepted_vintage()
print('shape:', accepted_clean.shape)
print(f"{accepted_clean.memory_usage(deep=True).sum() / 1e6:.0f} MB")
accepted_clean['loan_status'].value_counts(normalize=True)

## Doi chieu schema accepted vs rejected (RQ5)

Rejected chi co `Risk_Score` (xap xi FICO), accepted co `fico_range_low/high`. So sanh phan bo de danh gia muc do tuong dong truoc khi dung cho phan tich approval rate.

Rejected goc co 27.6M dong (~1.8GB) - doc toan bo se rat ton RAM khi accepted da nam san trong memory. `load_rejected_vintage()` doc theo chunk va loc luon ve cung vintage window (2015-2017) + chi giu cot can thiet, giam con ~15M dong / <1GB.

In [ ]:
rejected = load_rejected_vintage()
print(rejected.shape)
rejected.head()

In [ ]:
accepted_fico_mid = (accepted_clean['fico_range_low'] + accepted_clean['fico_range_high']) / 2
print('accepted fico_mid describe:')
print(accepted_fico_mid.describe())
print()
print('rejected Risk_Score describe:')
print(rejected['Risk_Score'].describe())

## Luu ra data/interim

In [ ]:
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
accepted_clean.to_parquet(DATA_INTERIM / 'accepted_vintage_2015_2017.parquet', index=False)
rejected.to_parquet(DATA_INTERIM / 'rejected_vintage_2015_2017.parquet', index=False)
print('done')